# AutoML · M04: H2O AutoML

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M04 — H2O AutoML |

---

## 🎯 Qué hace

AutoML distribuido con H2O: entrena GBM, XGBoost, DRF, GLM y StackedEnsemble
con validación cruzada 5-fold y optimización automática de hiperparámetros.
H2O destaca en entornos industriales (banca, seguros) por su robustez
y escalabilidad — su uso aquí valida que el enfoque del TFM es equiparable
a soluciones de nivel productivo.

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — 33.621 × 25 (generado por f3_m05)
- Java 8+ instalado en el sistema
- Entorno: `tfm_abandono`
- Librería: `h2o`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/results_h2o.parquet` | Métricas de ~20 modelos |
| `docs/html/fase_automl/m04_h2o.html` | Informe HTML |

## 🔄 Flujo

```
data/automl/dataset_final_tfm.parquet (24 features + abandono)
    ↓ H2OFrame + split 70/30
    ↓ H2OAutoML (max 10 min, CV=5, sort=AUC)
    ↓ Evaluación en test set
    → results_h2o.parquet + HTML
```

## ➡️ Siguiente

`fautoml_m05_autogluon.ipynb` — AutoGluon


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Auto-detecta Java (necesario para H2O), inicia servidor H2O,
# carga librerías del proyecto.
# ============================================================================

import sys
import warnings
import time
import subprocess
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Auto-detección de Java ---
def detectar_java():
    try:
        result = subprocess.run(['java', '-version'], capture_output=True, text=True)
        version_info = result.stderr.strip().split('\n')[0]
        print(f'✅ Java en PATH: {version_info}')
        return True
    except FileNotFoundError:
        pass

    # Buscar en ubicaciones típicas de Windows
    search_roots = [
        r'C:\Program Files\Eclipse Adoptium',
        r'C:\Program Files\Java',
        r'C:\Program Files\Temurin',
        r'C:\Program Files\Microsoft\jdk',
        r'C:\Program Files\Zulu',
    ]
    java_home = os.environ.get('JAVA_HOME')
    if java_home:
        bin_path = os.path.join(java_home, 'bin')
        if os.path.exists(bin_path):
            os.environ['PATH'] = bin_path + os.pathsep + os.environ['PATH']
            print(f'✅ Java via JAVA_HOME: {bin_path}')
            return True

    for root_dir in search_roots:
        if not os.path.exists(root_dir):
            continue
        for dirpath, dirnames, filenames in os.walk(root_dir):
            if 'java.exe' in filenames and os.path.basename(dirpath) == 'bin':
                os.environ['PATH'] = dirpath + os.pathsep + os.environ['PATH']
                try:
                    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
                    print(f'✅ Java auto-detectado: {dirpath}')
                    return True
                except Exception:
                    continue

    print('❌ Java no encontrado. Instalar desde: https://adoptium.net/')
    return False

if not detectar_java():
    raise RuntimeError('Java es necesario para H2O.')

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import h2o
from h2o.automl import H2OAutoML

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
RANDOM_STATE  = 42
FRAMEWORK     = 'h2o'
NOTEBOOK_NAME = 'fautoml_m04_h2o.ipynb'
CARPETA_NB    = 'fase_automl'

# Iniciar servidor H2O
h2o.init(max_mem_size='4G', nthreads=-1)
print(f'\n✅ H2O {h2o.__version__} iniciado')
info_entorno()
print('\n✅ Configuración completada')


✅ Java en PATH: openjdk version "21.0.10" 2026-01-20 LTS
✓ Directorios verificados: 2
.... not found.r there is an H2O instance running at http://localhost:54321.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-21.0.10+7 (build 21.0.10+7-LTS, mixed mode, sharing)
  Starting server from C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\mjmor\AppData\Local\Temp\tmpc768jfc0
  JVM stdout: C:\Users\mjmor\AppData\Local\Temp\tmpc768jfc0\h2o_mjmor_started_from_python.out
  JVM stderr: C:\Users\mjmor\AppData\Local\Temp\tmpc768jfc0\h2o_mjmor_started_from_python.err
  Server is running at http://127.0.0.1:54321
 successful.o H2O server at http://127.0.0.1:54321 ...
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Europe/Madrid
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,4 months and 19 days
H2O_cluster_name:,H2O_from_python_mjmor_3iszek
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.984 Gb
H2O_cluster_total_cores:,22
H2O_cluster_allowed_cores:,22
H2O_cluster_status:,"locked, healthy"



✅ H2O 3.46.0.9 iniciado
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ===

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y VERIFICACIÓN ANTI-LEAKAGE
# ============================================================================

print('\n' + '='*70)
print('CARGANDO DATASET CANÓNICO')
print('='*70)

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df = pd.read_parquet(ruta_dataset)

TARGET     = 'abandono'
n_total    = len(df)
n_features = len(df.columns) - 1
n_abandono = int(df[TARGET].sum())
tasa_ab    = n_abandono / n_total

print(f'\n✓ Dataset : {ruta_dataset.name}')
print(f'  Registros : {fmt(n_total)}')
print(f'  Features  : {n_features}')
print(f'  Abandono  : {fmt(n_abandono)} ({tasa_ab*100:.1f}%)')

# Verificación anti-leakage
COLS_LEAKAGE = [
    'egresado', 'egresado_de_hecho', 'curso_ultimo',
    'anos_inactivo', 'pct_titulacion', 'cred_faltantes',
    'progreso_relativo', 'titulacion', 'per_id_ficticio', 'exp_tit_id'
]
leakage_presente = [c for c in COLS_LEAKAGE if c in df.columns]
if leakage_presente:
    raise ValueError(f'LEAKAGE DETECTADO: {leakage_presente}')
print('\n✅ Verificación anti-leakage superada')

print(f'\n📋 Features ({n_features}):')
for i, col in enumerate([c for c in df.columns if c != TARGET], 1):
    print(f'  {i:2d}. {col}')



CARGANDO DATASET CANÓNICO

✓ Dataset : dataset_final_tfm.parquet
  Registros : 33.621
  Features  : 24
  Abandono  : 9.833 (29.2%)

✅ Verificación anti-leakage superada

📋 Features (24):
   1. cred_superados_anio_1er
   2. nota_1er_anio
   3. nota_acceso
   4. nota_selectividad
   5. via_acceso
   6. orden_preferencia
   7. cupo
   8. tasa_abandono_titulacion
   9. rama
  10. cred_repetidos
  11. tasa_repeticion
  12. sexo
  13. edad_entrada
  14. pais_nombre
  15. provincia
  16. universidad_origen
  17. n_anios_beca
  18. anios_sin_beca
  19. situacion_laboral
  20. n_anios_trabajando
  21. max_pagos
  22. indicador_interrupcion
  23. anios_gap
  24. n_anios_sin_notas


In [3]:
# ============================================================================
# CELDA 3: H2O AUTOML
# ============================================================================
# Carga datos en H2OFrame, split 70/30, ejecuta AutoML.
# max_runtime_secs=600 (10 min). Excluye DeepLearning para reducir tiempo.
# Evalúa cada modelo del leaderboard en test set independiente.
# ============================================================================

print('\n' + '='*70)
print('H2O AUTOML')
print('='*70)

# Convertir a H2OFrame
print('\n📤 Cargando datos en H2O...')
hf = h2o.H2OFrame(df)
hf[TARGET] = hf[TARGET].asfactor()   # clasificación binaria

# Split 70/30 estratificado
train, test = hf.split_frame(ratios=[0.7], seed=RANDOM_STATE)
features    = [c for c in hf.columns if c != TARGET]
print(f'  Train: {train.nrows:,} | Test: {test.nrows:,}')

# Ejecutar H2O AutoML
print(f'\n💧 Ejecutando H2O AutoML (max 10 min)...')
t0  = time.time()
aml = H2OAutoML(
    max_runtime_secs  = 600,
    max_models        = 20,
    seed              = RANDOM_STATE,
    sort_metric       = 'AUC',
    nfolds            = 5,
    verbosity         = 'warn',
    exclude_algos     = ['DeepLearning']
)
aml.train(x=features, y=TARGET, training_frame=train)
t_total = time.time() - t0

# Leaderboard
lb = aml.leaderboard.as_data_frame()
print(f'\n✅ {len(lb)} modelos entrenados en {t_total:.1f}s')
print(f'\n  Leaderboard (top 5):')
print(lb.head(5).to_string(index=False))

# Evaluar cada modelo en test set
print(f'\n🔄 Evaluando modelos en test set...')
all_results = []
for i, row_lb in lb.iterrows():
    model_id = row_lb['model_id']
    try:
        modelo = h2o.get_model(model_id)
        perf   = modelo.model_performance(test)

        # Nombre limpio
        model_name = model_id.split('_AutoML_')[0]
        if '_grid_' in model_name:
            model_name = model_name.split('_grid_')[0] + '_grid'

        # AUC y LogLoss — independientes del umbral, siempre fiables
        auc_val = perf.auc()     or 0
        ll_val  = perf.logloss() or 999

        # Precision, Recall, F1, MCC — recalculados con umbral 0.5 fijo
        # usando sklearn para comparabilidad con M01/M02/M03.
        # H2O reporta estas métricas en su umbral óptimo interno,
        # que puede diferir de 0.5 y producir valores artificialmente
        # perfectos (precision=1.0, recall=1.0). El umbral 0.5 es el
        # estándar en todos los demás módulos de esta fase AutoML.
        try:
            preds_h2o = modelo.predict(test).as_data_frame()
            y_prob    = preds_h2o['p1'].values          # probabilidad clase 1
            y_pred    = (y_prob >= 0.5).astype(int)       # umbral 0.5 fijo
            y_true    = test[TARGET].as_data_frame().values.ravel().astype(int)

            from sklearn.metrics import (
                accuracy_score, precision_score, recall_score,
                f1_score, matthews_corrcoef
            )
            acc_val  = accuracy_score(y_true, y_pred)
            prec_val = precision_score(y_true, y_pred, zero_division=0)
            rec_val  = recall_score(y_true, y_pred, zero_division=0)
            f1_val   = f1_score(y_true, y_pred, zero_division=0)
            mcc_val  = matthews_corrcoef(y_true, y_pred)
        except Exception as e_pred:
            print(f'    ⚠️ Predicción fallida para {model_name}: {e_pred}')
            acc_val  = 0.0
            prec_val = 0.0
            rec_val  = 0.0
            f1_val   = 0.0
            mcc_val  = 0.0

        all_results.append({
            'framework'        : FRAMEWORK,
            'model_name'       : model_name,
            'accuracy'         : acc_val,
            'balanced_accuracy': 0.0,
            'precision'        : prec_val,
            'recall'           : rec_val,
            'f1'               : f1_val,
            'auc_roc'          : auc_val,
            'mcc'              : mcc_val,
            'kappa'            : 0.0,
            'log_loss'         : round(ll_val, 4),
            'train_time_sec'   : round(t_total / max(len(lb), 1), 2),
        })
    except Exception as e:
        print(f'  ⚠️ Error con {model_id}: {e}')

df_resultados = (
    pd.DataFrame(all_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

print(f'\n--- RANKING FINAL (por F1) ---')
cols_show = ['model_name', 'f1', 'auc_roc', 'precision', 'recall', 'mcc']
print(df_resultados[cols_show].head(10).to_string(index=False))



H2O AUTOML

📤 Cargando datos en H2O...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
  Train: 23,645 | Test: 9,976

💧 Ejecutando H2O AutoML (max 10 min)...
AutoML progress: |
21:45:55.547: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"



✅ 21 modelos entrenados en 620.9s

  Leaderboard (top 5):
                                               model_id      auc  logloss    aucpr  mean_per_class_error     rmse      mse
StackedEnsemble_BestOfFamily_1_AutoML_1_20260412_214555 0.953968 0.238450 0.917902              0.120293 0.266472 0.071007
           GBM_grid_1_AutoML_1_20260412_214555_model_10 0.953879 0.239228 0.917891              0.119296 0.266864 0.071216
                         GBM_3_AutoML_1_20260412_214555 0.953667 0.238877 0.918194              0.121525 0.266050 0.070783
                         GBM_1_AutoML_1_20260412_214555 0.953588 0.239361 0.917513              0.119362 0.267062 0.071322
            GBM_grid_1_AutoML_1_20260412_214555_model_5 0.953066 0.240536 0.917633              0.117814 0.267272 0.071434

🔄 Evaluando modelos en test set...
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


...

C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
glm prediction progress: |

C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████████████████| (done) 100%

--- RANKING FINAL (por F1) ---
                    model_name       f1  auc_roc  precision   recall      mcc
                      GBM_grid 0.829554 0.954411   0.872979 0.790244 0.767226
                      GBM_grid 0.829402 0.954579   0.872643 0.790244 0.766985
                         GBM_4 0.829044 0.955450   0.877432 0.785714 0.767431
StackedEnsemble_BestOfFamily_1 0.828540 0.955213   0.867378 0.793031 0.765026
                         GBM_3 0.828498 0.954275   0.875776 0.786063 0.766480
                      GBM_grid 0.826207 0.953107   0.872819 0.784321 0.763224
                         GBM_1 0.825124 0.955345   0.869548 0.785017 0.761339
                      GBM_grid 0.823486 0.953439   0.866769 0.784321 0.758888
                      GBM_grid 0.822822 0.953875   0.862768 0.786411 0.757371
                         GBM_2 0.822560 0.954192   0.866847 0.782578 0.757804


C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [4]:
# ============================================================================
# CELDA 4: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_h2o.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} modelos guardados')


💾 results_h2o.parquet: 21 modelos guardados


In [5]:
# ============================================================================
# CELDA 5: GRÁFICOS Y HTML
# ============================================================================

graficos_b64 = {}

# --- Gráfico 1: Top 10 barras horizontales ---
df_plot = df_resultados.head(10).sort_values('f1', ascending=True).copy()
fig, ax = plt.subplots(figsize=(12, 7))
y_pos = np.arange(len(df_plot))
bars  = ax.barh(y_pos, df_plot['f1'], color='#805ad5', alpha=0.85, height=0.6)
ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=60, zorder=5, label='AUC-ROC')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['model_name'], fontsize=9)
ax.set_xlabel('Score')
ax.set_title('H2O AutoML: Top 10 Modelos', fontweight='bold', fontsize=14)
ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
ax.legend(fontsize=9)
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, df_plot['f1']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#2d3748')
plt.tight_layout()
graficos_b64['top10'] = figura_a_base64(fig)
plt.close()

# --- Gráfico 2: Top 5 barras agrupadas ---
df_top5   = df_resultados.head(5).copy()
metricas  = ['accuracy', 'f1', 'auc_roc', 'mcc']
etiquetas = ['Exactitud', 'F1', 'AUC-ROC', 'MCC']
colores   = ['#3182ce', '#38a169', '#e53e3e', '#805ad5']

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(df_top5))
width = 0.18
for j, (met, etiq, col) in enumerate(zip(metricas, etiquetas, colores)):
    ax.bar(x + j*width, df_top5[met], width, label=etiq, color=col)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(df_top5['model_name'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('H2O AutoML: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
graficos_b64['top5'] = figura_a_base64(fig)
plt.close()

# --- KPIs ---
mejor        = df_resultados.iloc[0]
nombre_mejor = mejor['model_name']
KPIS = [
    {'valor': fmt(n_total),            'titulo': 'Expedientes'},
    {'valor': str(n_features),         'titulo': 'Features'},
    {'valor': str(len(df_resultados)), 'titulo': 'Modelos evaluados'},
    {'valor': f'{mejor["f1"]:.4f}',    'titulo': f'Mejor F1 ({nombre_mejor})'},
]

# --- Sección metodología ---
s_metod = generar_seccion_html('Metodología', f'''
<p><strong>H2O AutoML {h2o.__version__}</strong> entrena {len(df_resultados)} modelos
(GBM, XGBoost, DRF, GLM, StackedEnsemble) con 5-fold CV y optimización
automática de hiperparámetros sobre {fmt(n_total)} expedientes y {n_features} features.</p>
<div style="background:#faf5ff;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #805ad5;">
  <strong>Relevancia industrial:</strong> H2O es ampliamente usado en banca
  y seguros para modelos de riesgo. Su uso aquí confirma que el enfoque
  del TFM es equiparable a soluciones de nivel productivo.
</div>
<div style="background:#ebf8ff;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #3182ce;">
  <strong>Config:</strong> max_runtime=600s, max_models=20, CV=5, sort=AUC,
  excluye DeepLearning. Métricas evaluadas en test set independiente (30%).
</div>''', 'ℹ️')

# --- Sección gráficos ---
graf_top10 = f'<img src="data:image/png;base64,{graficos_b64["top10"]}" style="max-width:100%;border-radius:8px;">'
graf_top5  = f'<img src="data:image/png;base64,{graficos_b64["top5"]}" style="max-width:100%;border-radius:8px;">'
s_graficos = generar_seccion_html('Top 10 modelos', graf_top10 + '<br>' + graf_top5, '📊')

# --- Tabla completa ---
tabla = '<table style="width:100%;border-collapse:collapse;font-size:11px;">\n'
tabla += '<tr style="background:#805ad5;color:white;">'
for col in ['#', 'Modelo', 'F1', 'AUC', 'Precision', 'Recall', 'MCC', 'LogLoss']:
    tabla += f'<th style="padding:8px;text-align:center;">{col}</th>'
tabla += '</tr>\n'
for rank, (i, row) in enumerate(df_resultados.iterrows(), 1):
    bg = '#f7fafc' if rank % 2 == 0 else 'white'
    medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
    rank_str = medallas.get(rank, str(rank))
    nombre_modelo = row['model_name']
    tabla += f'<tr style="background:{bg};">'
    tabla += f'<td style="padding:6px;text-align:center;">{rank_str}</td>'
    tabla += f'<td style="padding:6px;">{nombre_modelo}</td>'
    for campo in ['f1', 'auc_roc', 'precision', 'recall', 'mcc']:
        v = row[campo]
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'
    tabla += f'<td style="text-align:center;">{row["log_loss"]:.4f}</td>'
    tabla += '</tr>\n'
tabla += '</table>'
s_tabla = generar_seccion_html('Ranking completo de modelos', tabla, '🏆')

# --- Comparativa acumulada M01-M04 ---
s_comparativa = ''
frameworks_prev = []
for fname, flabel, tiene_cv in [
    ('results_baselines.parquet',   'M01 Baselines',          True),
    ('results_lazypredict.parquet', 'M02 LazyPredict',        False),
    ('results_pycaret.parquet',     'M03 PyCaret',            True),
]:
    ruta_fw = RUTA_AUTOML / fname
    if ruta_fw.exists():
        df_fw = pd.read_parquet(ruta_fw)
        no_dummy = ~df_fw['model_name'].str.startswith('Dummy')
        frameworks_prev.append((flabel, df_fw[no_dummy], tiene_cv))

if frameworks_prev:
    comp = '<table style="width:100%;border-collapse:collapse;">\n'
    comp += '<tr style="background:#805ad5;color:white;">'
    for col in ['Módulo', 'Mejor Modelo', 'F1', 'AUC', 'MCC', 'CV']:
        comp += f'<th style="padding:10px;text-align:center;">{col}</th>'
    comp += '</tr>\n'

    for fw_name, df_fw, cv_real in frameworks_prev:
        mejor_fw  = df_fw.sort_values('f1', ascending=False).iloc[0]
        nombre_fw = mejor_fw['model_name']
        cv_str    = 'CV=5 ✅' if cv_real else 'Sin CV ⚠️'
        comp += f'<tr style="background:#f7fafc;">'
        comp += f'<td style="padding:8px;">{fw_name}</td>'
        comp += f'<td style="padding:8px;">{nombre_fw}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["f1"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["auc_roc"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["mcc"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{cv_str}</td></tr>\n'

    nombre_h2o = mejor['model_name']
    comp += f'<tr style="background:white;font-weight:bold;">'
    comp += f'<td style="padding:8px;">M04 H2O AutoML</td>'
    comp += f'<td style="padding:8px;">{nombre_h2o}</td>'
    comp += f'<td style="text-align:center;">{mejor["f1"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor["auc_roc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor["mcc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">CV=5 ✅</td></tr>\n'
    comp += '</table>'
    s_comparativa = generar_seccion_html('Comparativa acumulada M01 → M04', comp, '🔄')

# --- Render HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm04_h2o.html'
render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_metod + s_graficos + s_tabla + s_comparativa,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)
print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m04_h2o.html


In [6]:
# ============================================================================
# CELDA 6: CERRAR H2O Y RESUMEN FINAL
# ============================================================================

# Cerrar servidor H2O — libera memoria y puerto
h2o.cluster().shutdown()

mejor = df_resultados.iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M04 COMPLETADO')
print('='*60)
print(f'Framework     : H2O {h2o.__version__}')
print(f'Dataset       : dataset_final_tfm.parquet ({fmt(n_total)} x {n_features+1})')
print(f'Validación    : 5-fold CV + test set independiente')
print(f'Modelos       : {len(df_resultados)}')
print(f'Mejor modelo  : {mejor["model_name"]}')
print(f'  F1          : {mejor["f1"]:.4f}')
print(f'  AUC         : {mejor["auc_roc"]:.4f}')
print(f'  MCC         : {mejor["mcc"]:.4f}')
print(f'Resultados    : {RUTA_AUTOML / "results_h2o.parquet"}')
print(f'HTML          : {RUTA_FASE_AUTOML_HTML / "m04_h2o.html"}')
print(f'\n➡️  Siguiente  : fautoml_m05_autogluon.ipynb')
print('='*60)


H2O session _sid_a462 closed.

✅ AUTOML-M04 COMPLETADO
Framework     : H2O 3.46.0.9
Dataset       : dataset_final_tfm.parquet (33.621 x 25)
Validación    : 5-fold CV + test set independiente
Modelos       : 21
Mejor modelo  : GBM_grid
  F1          : 0.8296
  AUC         : 0.9544
  MCC         : 0.7672
Resultados    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_h2o.parquet
HTML          : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m04_h2o.html

➡️  Siguiente  : fautoml_m05_autogluon.ipynb
